# EEG_11 — Dynamic Hypergraph Neural Network (D-HGNN)

**Motivazione EEG_10**: la CNN temporale impara feature soggetto-specifiche ma non generalizza.
Il vero problema non era la temporalità in sé, ma che **H era statico** — calcolato sull'intero
trial aggregato. Due elettrodi 'connessi in media' non significa che siano sincronizzati
nel momento cognitivo rilevante.

**Idea**: calcolare H_k **dinamicamente** per ogni finestra temporale — la topologia
dell'ipergrafo cambia nel tempo, catturando quando gli elettrodi si sincronizzano.
Questo è l'approccio alla base di DHSLP/DHSLF (Li et al. 2025, ~78% accuracy).

**Architettura D-HGNN**:
```
x (B, 61, 384)
  → split in K=8 finestre temporali
  per ogni finestra k (shared weights):
    x_k  (B, 61, 48)  → Linear → node_feat_k (B, 61, d_node=32)
    H_k  (B, 61, 61)  = |PCC(x_k)|  [calcolato on-the-fly, differenziabile]
    out_k (B, hidden) = HGNNConv(node_feat_k, H_k).mean(dim=1)
  z = mean(out_1, ..., out_K)  → (B, hidden)
  → Linear → logits (B, 4)
```

**Differenza chiave vs EEG_09/10**: H non viene caricato dal file .pt — viene calcolato
sul segnale grezzo di ogni finestra. La topologia dell'ipergrafo è una funzione del dato.

Ablation: 5 metriche (usa solo x dai file pruned, H ignorato) = 5 run.
Split: TRAIN sogg 0-49, VAL 50-59, TEST 60-73.

In [ ]:
import json, logging, re, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg11')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / '.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'

SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Dynamic HGNN
K_WINDOWS  = 8      # finestre temporali (384/8 = 48 sample = ~188ms a 256Hz)
D_NODE     = 32     # embedding per nodo per finestra (Linear 48->32)
HIDDEN     = 128
N_LAYERS   = 2
DROPOUT    = 0.3

# Training
LR             = 1e-3
BATCH_SIZE     = 64
MAX_EPOCHS     = 60
PATIENCE       = 12
USE_INSTANCE_NORM = True
LABEL_SMOOTHING   = 0.1

METRICS   = ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
LOAD_DIR  = 'hypergraphs_pruned'   # usa solo x e y dai file pruned

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}

T_WIN = N_SAMPLES // K_WINDOWS
log.info(f'K_WINDOWS={K_WINDOWS}  T_WIN={T_WIN}  D_NODE={D_NODE}  HIDDEN={HIDDEN}')
log.info(f'SUBJ TRAIN={len(SUBJ_TRAIN)} VAL={len(SUBJ_VAL)} TEST={len(SUBJ_TEST)}')


## §2 — Dataset (carica solo x e y, H calcolato dinamicamente)

In [ ]:
class EEGWindowDataset(Dataset):
    """Carica x (61,384) e y dai file .pt. H ignorato — calcolato on-the-fly nel modello."""
    def __init__(self, subj_ids, metric, use_instance_norm=True):
        root = project_root / 'data' / f'{LOAD_DIR}_{metric}'
        self.paths, self.labels = [], []
        self.use_instance_norm = use_instance_norm
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            if int(m.group(1)) not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p)
            self.labels.append(c)
        log.info(f'  {LOAD_DIR}_{metric}: {len(self.paths)} trial')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()   # (61, 384)
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def make_loaders(metric):
    tr = EEGWindowDataset(SUBJ_TRAIN, metric, USE_INSTANCE_NORM)
    va = EEGWindowDataset(SUBJ_VAL,   metric, USE_INSTANCE_NORM)
    te = EEGWindowDataset(SUBJ_TEST,  metric, USE_INSTANCE_NORM)
    kw = dict(num_workers=2, pin_memory=True)
    labels  = np.array(tr.labels)
    counts  = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / counts[labels], dtype=torch.float)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, BATCH_SIZE, shuffle=False, **kw))


## §3 — Modello: D-HGNN

In [ ]:
def dynamic_H(x_win):
    """
    H_k = |PCC(x_win)| — differenziabile, gira su GPU.
    x_win: (B, N, T_k) -> H: (B, N, N) in [0,1]
    """
    x_c = x_win - x_win.mean(dim=2, keepdim=True)
    norm = x_c.norm(dim=2, keepdim=True).clamp(min=1e-6)
    x_n  = x_c / norm
    return torch.bmm(x_n, x_n.transpose(1, 2)).abs()


class HGNNConv(nn.Module):
    """HGNN layer (Feng et al. 2019)"""
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6)
        d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv  = (1.0 / d_v.sqrt()).unsqueeze(-1)
        De  = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1,2), out)
        out = De.transpose(1,2) * out
        out = torch.bmm(H, out)
        out = Dv * out
        if self.bias is not None: out = out + self.bias
        return out


class DynamicHGNN(nn.Module):
    """
    Per ogni finestra k:
      node_feat_k = Linear(x_k)   (B,N,T_win) -> (B,N,d_node)
      H_k         = |PCC(x_k)|   (B,N,N)  — topologia dinamica
      out_k       = HGNN(node_feat_k, H_k).mean(dim=1)  (B, hidden)
    z = mean(out_1..K) -> clf
    """
    def __init__(self, T_win=T_WIN, K=K_WINDOWS, d_node=D_NODE,
                 hidden=HIDDEN, n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win = K, T_win
        self.node_proj = nn.Sequential(
            nn.Linear(T_win, d_node),
            nn.LayerNorm(d_node),
            nn.ELU(),
        )
        dims = [d_node] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden, n_classes)

    def forward(self, x):
        B, N, T = x.shape
        outs = []
        for k in range(self.K):
            x_k  = x[:, :, k*self.T_win:(k+1)*self.T_win]  # (B,N,T_win)
            H_k  = dynamic_H(x_k)                           # (B,N,N)
            feat = self.node_proj(x_k)                      # (B,N,d_node)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out)
                out = self.drop(out)
            outs.append(out.mean(dim=1))                    # (B, hidden)
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))


# Sanity check
_m = DynamicHGNN()
_x = torch.randn(4, N_CHANNELS, N_SAMPLES)
assert _m(_x).shape == (4, N_CLASSES)
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'DynamicHGNN OK — {n_p:,} parametri  K={K_WINDOWS} x {T_WIN} sample/finestra')
del _m, _x


## §4 — Train / Eval

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')

_criterion = None


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = _criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_labels.extend(y.cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
    bacc = balanced_accuracy_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), bacc, np.array(all_labels), np.array(all_preds)


def train_model(run_name, tr_loader, va_loader, te_loader, config):
    global _criterion
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=run_name, config=config, reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))
    _criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    model = DynamicHGNN().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, patience_cnt = 0.0, None, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_bacc, _, _ = run_epoch(model, tr_loader, opt)
        va_loss, va_bacc, _, _ = run_epoch(model, va_loader)
        sched.step()
        run.log({'train/loss': tr_loss, 'train/bacc': tr_bacc,
                 'val/loss': va_loss,   'val/bacc': va_bacc, 'epoch': epoch})
        if va_bacc > best_val:
            best_val = va_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            log.info(f'  early stop epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, te_bacc, te_labels, te_preds = run_epoch(model, te_loader)
    run.summary['val_bacc']  = best_val
    run.summary['test_bacc'] = te_bacc
    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
        preds=te_preds.tolist(), y_true=te_labels.tolist(),
        class_names=['CONCR', 'AZIONE', 'STATO', 'ASTRATTO'])})
    run.finish()
    log.info(f'  {run_name}: val={best_val:.4f} test={te_bacc:.4f}')
    return best_val, te_bacc, te_labels, te_preds


## §5 — Ablation Loop (5 metriche)

In [ ]:
RESULTS = {}

for metric in METRICS:
    name = f'eeg11_DHGNN_{metric}_{CLUSTER_SCHEME}'
    log.info(f'\n=== {name} ===')
    try:
        tr_l, va_l, te_l = make_loaders(metric)
    except Exception as e:
        log.warning(f'  Skip {metric}: {e}'); continue

    cfg = dict(
        notebook='EEG_11', model='DynamicHGNN', metric=metric,
        n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
        k_windows=K_WINDOWS, t_win=T_WIN, d_node=D_NODE,
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
        lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
        use_instance_norm=USE_INSTANCE_NORM,
        label_smoothing=LABEL_SMOOTHING,
        weighted_sampler=True,
        dynamic_H='abs_pcc_per_window',
        n_train_subj=len(SUBJ_TRAIN),
    )
    val_b, test_b, lbl, pred = train_model(name, tr_l, va_l, te_l, cfg)
    RESULTS[metric] = {'val': val_b, 'test': test_b}

log.info('\n=== ABLATION DONE ===')
for k, v in RESULTS.items():
    print(f'  {k:15s}  val={v["val"]:.4f}  test={v["test"]:.4f}')


## §6 — Risultati + Confronto EEG_09 / EEG_10

In [ ]:
EEG09 = {'pcc': 0.260, 'abs_pcc': 0.253, 'im_pcc': 0.256, 'wpli': 0.255, 'plv': 0.260}
EEG10 = {'pcc': 0.248, 'abs_pcc': 0.252, 'im_pcc': 0.250, 'wpli': 0.249, 'plv': 0.241}

if RESULTS:
    chance = 1 / N_CLASSES
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('EEG_11 — D-HGNN Dynamic Hypergraph (bAcc)', fontsize=13, fontweight='bold')

    for ax, split in zip(axes, ['val', 'test']):
        v11 = [RESULTS.get(m, {}).get(split, np.nan) for m in METRICS]
        y_pos = np.arange(len(METRICS))
        h = 0.25
        ax.barh(y_pos + h, v11, height=h, label='D-HGNN (EEG_11)', color='#1565C0')
        if split == 'test':
            v10 = [EEG10.get(m, np.nan) for m in METRICS]
            v09 = [EEG09.get(m, np.nan) for m in METRICS]
            ax.barh(y_pos,     v10, height=h, label='T-HGNN (EEG_10)',      color='#42A5F5')
            ax.barh(y_pos - h, v09, height=h, label='HGNN static (EEG_09)', color='#90CAF9')
        ax.axvline(chance, color='gray', linestyle='--', linewidth=1.2, label=f'Chance ({chance:.2f})')
        ax.set_yticks(y_pos); ax.set_yticklabels(METRICS, fontsize=10)
        ax.set_xlim(0.18, max(0.55, max([v for v in v11 if not np.isnan(v)] or [0.55]) + 0.06))
        ax.set_xlabel('Balanced Accuracy')
        ax.set_title(f'{split.capitalize()} bAcc', fontsize=11)
        ax.legend(fontsize=8, loc='lower right')
        for i, v in enumerate(v11):
            if not np.isnan(v):
                ax.text(v + 0.003, y_pos[i] + h/2, f'{v:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eeg11_dhgnn_ablation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n--- Test bAcc: D-HGNN vs baseline ---')
    rows = []
    for m in METRICS:
        v11t = RESULTS.get(m, {}).get('test', np.nan)
        rows.append({'metric': m,
                     'eeg09': EEG09.get(m, np.nan),
                     'eeg10': EEG10.get(m, np.nan),
                     'eeg11': v11t,
                     'delta_vs_09': round(v11t - EEG09.get(m, np.nan), 4) if not np.isnan(v11t) else np.nan})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    df.to_csv(FIG_DIR / 'eeg11_dhgnn_results.csv', index=False)
